In [ ]:
!git clone https://github.com/aria-yang/youtube-thumbnail-performance-predictor.git
%cd /content/youtube-thumbnail-performance-predictor


Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q numpy pandas pillow tqdm scikit-learn loguru typer python-dotenv
!pip install -q easyocr facenet-pytorch deepface

Run full merged pipeline

In [ ]:
from google.colab import files

uploaded = files.upload()
artifact_zip_name = next(iter(uploaded))
print(f"Uploaded {artifact_zip_name}")

Artifact upload note

Download the shared artifact bundle as a single zip file, upload it in the previous cell, and this notebook will unpack it directly into the local Colab repo.

In [ ]:
import shutil
import zipfile
from pathlib import Path

repo_root = Path("/content/youtube-thumbnail-performance-predictor")
upload_path = Path("/content") / artifact_zip_name
extract_root = Path("/content/artifact_bundle")

if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(upload_path, "r") as archive:
    archive.extractall(extract_root)

processed_dir = repo_root / "data" / "processed"
splits_dir = repo_root / "data" / "splits"
models_dir = repo_root / "models"
for directory in [processed_dir, splits_dir, models_dir]:
    directory.mkdir(parents=True, exist_ok=True)

split_names = {
    "random_train.csv", "random_val.csv", "random_test.csv",
    "channel_train.csv", "channel_val.csv", "channel_test.csv",
    "time_train.csv", "time_val.csv", "time_test.csv",
}

for src_path in extract_root.rglob("*"):
    if not src_path.is_file():
        continue
    name = src_path.name
    if name.endswith(".pt"):
        dst_path = models_dir / name
    elif name in split_names:
        dst_path = splits_dir / name
    else:
        dst_path = processed_dir / name
    shutil.copy2(src_path, dst_path)
    print(f"Copied {name} -> {dst_path.relative_to(repo_root)}")

In [ ]:
!PYTHONPATH=. python training/train_fusion_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss smoothl1 \
  --num_epochs 50 \
  --batch_size 128

## Tuning

In [ ]:
!PYTHONPATH=. python training/tune_fusion_regression.py \
  --device auto \
  --split_names random \
  --target_modes log1p,log1p_zscore \
  --losses smoothl1,mse \
  --batch_sizes 64,128 \
  --lrs 0.0005,0.001 \
  --dropouts 0.3,0.4 \
  --hidden_dims 512x256,1024x512 \
  --seeds 42,43 \
  --num_epochs 20 \
  --metric_to_rank val_spearman

## Final Training

In [ ]:
!PYTHONPATH=. python training/train_fusion_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --lr 0.001 \
  --num_epochs 60 \
  --seed 42 \
  --checkpoint_path models/fusion_mlp_regression_final_seed42.pt \
  --metrics_path data/processed/fusion_mlp_regression_final_seed42_metrics.json

#Ablation and SHAP

In [ ]:
!PYTHONPATH=. python training/ablation_study_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --num_epochs 60 \
  --lr 0.001 \
  --hidden1 512 \
  --hidden2 256 \
  --dropout 0.4 \
  --ranking_metric test_spearman \
  --output_dir outputs/ablation_regression

In [ ]:
!PYTHONPATH=. python training/run_shap_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --num_epochs 60 \
  --lr 0.001 \
  --hidden1 512 \
  --hidden2 256 \
  --dropout 0.4 \
  --seed 42 \
  --checkpoint_path models/fusion_mlp_regression_final_seed42.pt \
  --output_dir outputs/shap_seed42

## Cross-Split Evaluation

In [ ]:
!PYTHONPATH=. python training/eval_crosssplit_regression.py \
  --device auto \
  --target_transform log1p \
  --checkpoint_path models/fusion_mlp_regression_final_seed42.pt \
  --output_dir outputs/cross_split_regression